In [1]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

np.random.seed(99)

In [2]:

def generate_random_individuals(population_size, num_features, barcode_size):
    individuals = np.zeros((population_size, num_features))
    for i in range(population_size):
        num_ones = barcode_size
        ones_indices = np.random.choice(num_features, num_ones, replace=False)
        individuals[i, ones_indices] = 1
    return individuals

In [3]:

def train_model(X , Y, predictor_names):
    
    X_current = X.loc[:, predictor_names]
    
    splits = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    mdl = SVC(kernel = 'linear', class_weight="balanced",
                        max_iter=1000, random_state=1)

    scores = cross_val_score(mdl, X_current, Y, cv=splits, scoring="f1")
    return scores.mean()

In [4]:
def choose_parents(population,accuracy):
    elite_num = 1
    ind_ac = np.argsort(-accuracy)
    top_perc = ind_ac[:elite_num]
    elite_population = population[top_perc,:]  
    
    n = len(accuracy)
    ranks = np.empty(n)
    ranks[ind_ac] = np.arange(n, 0, -1)   
    weight_comu = (ranks / ranks.sum()).cumsum()       
     
    num_parents_wo_elite = population.shape[0] - elite_num
    parents_wo_elite = np.empty([num_parents_wo_elite,population.shape[1]])
    for count in range(num_parents_wo_elite):
        b = weight_comu[-1]                       
        rand_num = np.random.uniform(0,b)      
        
        indices = np.searchsorted(weight_comu,rand_num) 
        parents_wo_elite[count,:] = population[indices, :]
        
    parents = np.concatenate((elite_population, parents_wo_elite), axis=0) 
    return parents

In [5]:
def one_point_crossover(parents, mutation_probability, barcode_size):
    elite_num = 1
    crossover_population = np.zeros((parents.shape[0]-elite_num,parents.shape[1]))  
    
    for ii in range(int((parents.shape[0]-elite_num)/2)):
        n = 2*ii+elite_num                   
        parents_couple = parents[n:n+2, :]   
        b2 = parents.shape[1]               
        rand_n = np.random.randint(1, b2-1)   
        crossover_population[n-1,:] = np.concatenate([parents_couple[0, :rand_n], parents_couple[1, rand_n:]])
        crossover_population[n,:] = np.concatenate([parents_couple[1, :rand_n], parents_couple[0, rand_n:]])

    # mutation
    child_row = crossover_population.shape[0]
    child_col = crossover_population.shape[1]
    num_mutations = round(child_row*child_col*mutation_probability)     
    for jj in range(num_mutations):
        ind_row = np.random.randint(0,child_row) 
        ind_col = np.random.randint(0,child_col) 
        if crossover_population[ind_row,ind_col] == 0:
            crossover_population[ind_row,ind_col] = 1
        elif crossover_population[ind_row,ind_col] == 1:
            crossover_population[ind_row,ind_col] = 0 
        
    elite_parents = parents[0:elite_num,:]
    crossover_population = np.concatenate([elite_parents, crossover_population], axis=0)
    
    for kk in range(crossover_population.shape[0]):
        Sum = np.sum(crossover_population[kk,:])
        if Sum > barcode_size:
            
            excess = int(Sum - barcode_size)
            indices = np.where(crossover_population[kk,:] == 1)[0]
            position1 = np.random.choice(indices, size=excess, replace=False)       
            crossover_population[kk, position1] = 0 
        elif Sum < barcode_size:
    
            missing = int(barcode_size - Sum)
            indices = np.where(crossover_population[kk,:] == 0)[0]
            position2 = np.random.choice(indices, size=missing, replace=False) 
            crossover_population[kk, position2] = 1 
    
    return crossover_population

In [6]:
data = pd.read_csv('train_barcode.csv')
data = data.set_index('Sample')

target = data.loc[:, 'Species']
target = target.replace({"Pk": 1, "Pv": 0}).astype(int) 
data_predictors = data.loc[:, data.columns != 'Species']
predictor_names = data_predictors.columns
population_size = 201
max_iterations = 300

/tmp/ipykernel_595/253518186.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  target = target.replace({"Pk": 1, "Pv": 0}).astype(int)


In [7]:
barcode_lengths = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
stop_points = []

for barcode_size in barcode_lengths:
    population = generate_random_individuals(population_size, data_predictors.shape[1], barcode_size)
    accuracy = np.zeros(population_size)
    for i in range(population_size):
        accuracy[i] = train_model(X=data_predictors, Y=target, predictor_names=predictor_names[population[i, :] == 1])

    best_acc_i = np.zeros(max_iterations)   
    best_acc_i[0] = accuracy.max()
    gen = 0

    while gen < max_iterations - 1 and accuracy.max() < 1.0:
        gen += 1
        parents = choose_parents(population=population, accuracy=accuracy)
        population = one_point_crossover(parents=parents, mutation_probability=0.0005, barcode_size=barcode_size)
        for ind in range(population_size):
            accuracy[ind] = train_model(X=data_predictors, Y=target, predictor_names=predictor_names[population[ind, :] == 1])
        best_acc_i[gen] = accuracy.max()      

    best_acc_i = best_acc_i[:gen + 1]        
    best_names = predictor_names[population[np.argmax(accuracy), :] == 1]  

    with open(f"GA_subsets_2/top_{barcode_size}_GA.txt", "w") as f:
        f.write("\n".join(best_names))

    stop_points.append({"Barcode size": barcode_size, "Generations": gen + 1})
    pd.DataFrame(stop_points).to_csv("GA_subsets_2/stop_points.csv", index=False)